In [1]:

# ============================================================
# CELL 1: Install dependencies
# ============================================================
!pip install -q transformers torch pandas scikit-learn


In [2]:

# ============================================================
# CELL 2: Imports
# ============================================================
import os
import re
import json
import glob
import shutil
import zipfile
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import pandas as pd
from transformers import AutoTokenizer, AutoModel, AutoConfig
from transformers.modeling_outputs import ModelOutput
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    mean_squared_error,
    mean_absolute_error,
    classification_report,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")



Using device: cuda


In [ ]:
!ls /content/drive/MyDrive/

ls: cannot access '/content/drive/MyDrive/': No such file or directory


In [3]:

# ============================================================
# CELL 3: Load the trained model checkpoint from Google Drive
#
# files.upload() (the browser upload widget) is unreliable for large
# files like a ~400MB model checkpoint — it can stall for a long time
# with no progress feedback. Uploading to Drive first (drag-and-drop
# in your browser, which is fast and resumable) and mounting Drive
# here instead is far more reliable.
#
# Before running this cell:
#   1. Upload your trained model zip (the zipped "best/" directory —
#      containing config.json, pytorch_model.bin, tokenizer files,
#      and training_meta.json) to your Google Drive, e.g. into
#      "My Drive/colab_models/".
#   2. Set MODEL_ZIP_DRIVE_PATH below to that file's path.
# ============================================================
from google.colab import drive

drive.mount("/content/drive")

MODEL_ZIP_DRIVE_PATH = "/content/drive/MyDrive/roberta-base_model.zip"


assert os.path.exists(MODEL_ZIP_DRIVE_PATH), (
    f"Model zip not found at {MODEL_ZIP_DRIVE_PATH} — check the path matches "
    "where you uploaded it in your Google Drive."
)

MODEL_ZIP_PATH = "/content/eval_model.zip"
shutil.copy(MODEL_ZIP_DRIVE_PATH, MODEL_ZIP_PATH)
print(f"Copied model checkpoint from Drive ({os.path.getsize(MODEL_ZIP_PATH) / 1e6:.1f} MB).")

MODEL_DIR = "/content/eval_model"
if os.path.exists(MODEL_DIR):
    shutil.rmtree(MODEL_DIR)
os.makedirs(MODEL_DIR, exist_ok=True)

with zipfile.ZipFile(MODEL_ZIP_PATH, "r") as zf:
    zf.extractall(MODEL_DIR)

# The zip may extract straight into MODEL_DIR, or into a single
# subfolder (e.g. "best/") depending on how it was zipped — detect
# and flatten either layout so config.json is always at MODEL_DIR root.
if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    subdirs = [
        d for d in glob.glob(os.path.join(MODEL_DIR, "*"))
        if os.path.isdir(d)
    ]
    if len(subdirs) == 1 and os.path.exists(os.path.join(subdirs[0], "config.json")):
        for item in os.listdir(subdirs[0]):
            shutil.move(os.path.join(subdirs[0], item), MODEL_DIR)
        shutil.rmtree(subdirs[0])

assert os.path.exists(os.path.join(MODEL_DIR, "config.json")), (
    f"config.json not found in {MODEL_DIR} — check the uploaded zip contains "
    "the checkpoint files directly (config.json, pytorch_model.bin, tokenizer files)."
)
print(f"Model checkpoint ready at: {MODEL_DIR}")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied model checkpoint from Drive (416.2 MB).
Model checkpoint ready at: /content/eval_model


In [4]:

# ============================================================
# CELL 4: Upload dataset CSV(s) and lexicon.json
#
# Upload one or more dataset_match_*.csv files (multi-select is fine)
# AND commentary_analysis_system/data/lexicon.json. These are small
# text files (a few MB at most), so the files.upload() widget is fine
# here — it's only large binaries like the model checkpoint that need
# the Drive-mount approach used in Cell 3.
# ============================================================
from google.colab import files

print("Upload one or more dataset_match_*.csv files, plus lexicon.json:")
uploaded_data = files.upload()

DATASET_CSV_PATHS = sorted(
    name for name in uploaded_data.keys() if name.lower().endswith(".csv")
)
# Colab's upload widget appends " (1)", " (2)", etc. to the saved filename
# whenever /content already has a file with that exact name (e.g. left over
# from a previous run in this session) — match that pattern too, not just
# the exact "lexicon.json", or LEXICON_JSON_PATH silently ends up None.
LEXICON_JSON_PATH = next(
    (
        name for name in uploaded_data.keys()
        if re.match(r"^lexicon(?: \(\d+\))?\.json$", os.path.basename(name), re.IGNORECASE)
    ),
    None,
)

assert DATASET_CSV_PATHS, "No CSV files were uploaded."
assert LEXICON_JSON_PATH, (
    "lexicon.json was not uploaded (or was saved under an unexpected name — "
    f"got: {sorted(uploaded_data.keys())})."
)
print(f"Datasets ({len(DATASET_CSV_PATHS)}): {DATASET_CSV_PATHS}")
print(f"Lexicon config: {LEXICON_JSON_PATH}")



Upload one or more dataset_match_*.csv files, plus lexicon.json:


Saving dataset_match_1_Geo_vs_POR_WCA_2023.csv to dataset_match_1_Geo_vs_POR_WCA_2023 (1).csv
Saving dataset_match_2_NZ_vs_ITA_WCA_2023.csv to dataset_match_2_NZ_vs_ITA_WCA_2023 (1).csv
Saving dataset_match_3_Aus_vs_Fij_WCA_2023.csv to dataset_match_3_Aus_vs_Fij_WCA_2023 (1).csv
Saving dataset_match_4_Eng_vs_Arg_PoolD_WCA_2023.csv to dataset_match_4_Eng_vs_Arg_PoolD_WCA_2023 (1).csv
Saving dataset_match_5_Ire_vs_Scot_poolB_2023.csv to dataset_match_5_Ire_vs_Scot_poolB_2023 (1).csv
Saving dataset_match_6_SA_vs_IRE_WCA_2023.csv to dataset_match_6_SA_vs_IRE_WCA_2023 (1).csv
Saving dataset_match_7_jpn_vs_sam_ZRWC_2023.csv to dataset_match_7_jpn_vs_sam_ZRWC_2023 (1).csv
Saving dataset_match_8_Fra_vs_NZL_RWC_2023.csv to dataset_match_8_Fra_vs_NZL_RWC_2023 (1).csv
Saving dataset_match_9_Jpn_vs_Arg_RWC_2023.csv to dataset_match_9_Jpn_vs_Arg_RWC_2023 (1).csv
Saving lexicon.json to lexicon (1).json
Datasets (9): ['dataset_match_1_Geo_vs_POR_WCA_2023 (1).csv', 'dataset_match_2_NZ_vs_ITA_WCA_2023 

In [5]:

# ============================================================
# CELL 5: DualHead model definition
#
# Plain nn.Linear heads — matches colab_backend.py / colab
# testing/colab_training.py (the scripts that actually produced every
# trained checkpoint in this project), not the Sequential(Dropout,
# Linear) variant used by the local pipeline's transformer_classifier.py.
# ============================================================
@dataclass
class DualHeadRoBERTaOutput(ModelOutput):
    loss: Optional[torch.FloatTensor] = None
    logits: torch.FloatTensor = None
    highlight_scores: torch.FloatTensor = None


class DualHeadRoBERTa(nn.Module):
    def __init__(self, model_name, num_labels, id2label, label2id):
        super().__init__()
        self.config = AutoModel.from_pretrained(model_name).config
        self.config.num_labels = num_labels
        self.config.id2label = id2label
        self.config.label2id = label2id

        self.roberta = AutoModel.from_pretrained(model_name)

        self.event_classifier = nn.Linear(self.config.hidden_size, num_labels)
        self.highlight_scorer = nn.Linear(self.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask, **kwargs):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = getattr(outputs, "pooler_output", None)
        if pooled_output is None:
            pooled_output = outputs.last_hidden_state[:, 0, :]

        event_logits = self.event_classifier(pooled_output)
        highlight_score = torch.sigmoid(self.highlight_scorer(pooled_output))

        return DualHeadRoBERTaOutput(logits=event_logits, highlight_scores=highlight_score)



In [6]:

# ============================================================
# CELL 6: Load the checkpoint
#
# Reads id2label/label2id/num_labels from config.json, and
# window_size/tokenizer_max_length from training_meta.json (falling
# back to config attributes, then to window_size=1 /
# tokenizer_max_length=128 if neither is present).
# ============================================================
class LoadedModel:
    def __init__(self, model_dir):
        config = AutoConfig.from_pretrained(model_dir)
        self.num_labels = config.num_labels
        self.id2label = {int(k): v for k, v in config.id2label.items()}
        self.label2id = {k: int(v) for k, v in config.label2id.items()}

        self.window_size = getattr(config, "window_size", 1)
        self.tokenizer_max_length = getattr(config, "tokenizer_max_length", 128)

        meta_path = os.path.join(model_dir, "training_meta.json")
        if os.path.exists(meta_path):
            with open(meta_path, "r") as f:
                meta = json.load(f)
            self.window_size = meta.get("window_size", self.window_size)
            self.tokenizer_max_length = meta.get("tokenizer_max_length", self.tokenizer_max_length)

        base_model_name = getattr(config, "_name_or_path", "roberta-base")
        self.model = DualHeadRoBERTa(base_model_name, self.num_labels, self.id2label, self.label2id)

        state_dict = torch.load(os.path.join(model_dir, "pytorch_model.bin"), map_location="cpu")
        self.model.load_state_dict(state_dict)
        self.model.to(DEVICE)
        self.model.eval()

        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)

    def predict_probs(self, text_list, batch_size=32):
        """Returns (event_probs, highlight_scores) — one list of 10-way
        softmax probs and one highlight-score float per input text."""
        all_event_probs, all_highlight_scores = [], []
        for i in range(0, len(text_list), batch_size):
            batch_texts = text_list[i:i + batch_size]
            inputs = self.tokenizer(
                batch_texts, padding=True, truncation=True,
                max_length=self.tokenizer_max_length, return_tensors="pt",
            )
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = self.model(**inputs)
                probs = torch.softmax(outputs.logits, dim=-1)
                all_event_probs.extend(probs.tolist())
                all_highlight_scores.extend(outputs.highlight_scores.view(-1).tolist())
        return all_event_probs, all_highlight_scores


loaded_model = LoadedModel(MODEL_DIR)
print(f"Loaded model: num_labels={loaded_model.num_labels}, "
      f"window_size={loaded_model.window_size}, "
      f"tokenizer_max_length={loaded_model.tokenizer_max_length}")
print(f"id2label: {loaded_model.id2label}")



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: /content/eval_model
Key                     | Status     |  | 
------------------------+------------+--+-
event_classifier.bias   | UNEXPECTED |  | 
event_classifier.weight | UNEXPECTED |  | 
highlight_scorer.bias   | UNEXPECTED |  | 
highlight_scorer.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: /content/eval_model
Key                     | Status     |  | 
------------------------+------------+--+-
event_classifier.bias   | UNEXPECTED |  | 
event_classifier.weight | UNEXPECTED |  | 
highlight_scorer.bias   | UNEXPECTED |  | 
highlight_scorer.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: num_labels=10, window_size=5, tokenizer_max_length=256
id2label: {0: 'normal_play', 1: 'try', 2: 'goal_kick', 3: 'card_event', 4: 'scrum', 5: 'maul', 6: 'lineout', 7: 'kick_off', 8: 'tmo_replay', 9: 'penalty'}


In [7]:

# ============================================================
# CELL 7: Sliding-window context builder
#
# Ported from commentary_analysis_system/src/data_processing/chunker.py.
# The ML/Hybrid model was trained on windowed text (each row joined
# with its neighbors) — skipping this for a checkpoint with
# window_size > 1 collapses predictions to a single class.
# ============================================================
def build_context_windows(texts: list[str], window_size: int) -> list[str]:
    if window_size <= 1:
        return list(texts)
    half = window_size // 2
    n = len(texts)
    return [
        " ".join(texts[max(0, i - half): min(n, i + half + 1)])
        for i in range(n)
    ]



In [8]:

# ============================================================
# CELL 8: Lexicon model
#
# Ported verbatim from
# commentary_analysis_system/src/models/lexicon_model.py.
# ============================================================
class LexiconModel:
    def __init__(self, config_path):
        with open(config_path, "r") as f:
            self.config = json.load(f)

    def _count_matches_weighted(self, text: str, terms: list[dict]) -> float:
        score = 0.0
        negations = [
            "no", "not", "missed", "missed the",
            "almost", "nearly", "just", "prevented",
            "stopped", "denied", "failed", "attempt",
            "trying to", "going for", "couldn't", "could not",
            "short of", "held up", "short",
        ]
        for term_obj in terms:
            if isinstance(term_obj, str):
                term, weight = term_obj, 1.0
            else:
                term = term_obj.get("text", "")
                weight = float(term_obj.get("weight", 1.0))
            if not term:
                continue

            if "\\" in term:
                pattern = term
            else:
                escaped_term = re.escape(term)
                pattern = rf"\b{escaped_term}(?:s|ed|ing|d)?\b"

            for match in re.finditer(pattern, text, re.IGNORECASE):
                start_idx = match.start()
                preceding_text = text[max(0, start_idx - 15):start_idx].lower()
                is_negated = any(preceding_text.strip().endswith(neg) for neg in negations)
                if not is_negated:
                    score += weight
        return score

    def generate_features(self, text: str) -> dict:
        text = text.lower()
        features = {}
        for category in self.config.get("categories", []):
            cat_id = category["id"]
            features[cat_id] = self._count_matches_weighted(text, category.get("terms", []))
        return features

    def score_chunk(self, text: str) -> float:
        feats = self.generate_features(text)
        raw_score = 0.0
        for category in self.config.get("categories", []):
            cat_id = category["id"]
            weight = category.get("weight", 0.0)
            count = feats.get(cat_id, 0)
            scaled_count = float(count) * 100.0
            raw_score += min(1.0, scaled_count) * weight
        return min(raw_score, 1.0)


lexicon_model = LexiconModel(LEXICON_JSON_PATH)
print(f"Loaded lexicon with {len(lexicon_model.config.get('categories', []))} categories.")



Loaded lexicon with 10 categories.


In [9]:
# ============================================================
# CELL 9: Hybrid combination (confidence-gated)
#
# The lexicon only gets a say when RoBERTa itself is uncertain — i.e.
# when the gap between its top-1 and top-2 class probabilities is
# below MARGIN_THRESHOLD. When RoBERTa is confident (a wide margin),
# hybrid_pred is set to ml_pred exactly, so on that subset hybrid
# cannot regress relative to ML-only — it's the same prediction.
#
# This does not give an absolute mathematical guarantee that hybrid
# accuracy >= ML-only accuracy overall (within the "uncertain" zone
# the lexicon can still occasionally flip a correct call to wrong),
# but it removes the risk entirely from every row where RoBERTa was
# already confident, which is where most of the previous small
# regression was coming from.
# ============================================================
HIGHLIGHT_LEXICON_WEIGHT = 0.75
EVENT_LEXICON_WEIGHT = 0.20
ROBERTA_WEIGHT = 1.0
MARGIN_THRESHOLD = 0.05  # gate: only let lexicon influence event_id when top1-top2 < this

CATEGORY_WEIGHTS = {c["id"]: c["weight"] for c in lexicon_model.config["categories"]}


def run_all_approaches(texts: list[str], id2label: dict, model: LoadedModel, lexicon: LexiconModel):
    """
    texts: raw per-row commentary text, in original CSV row order.
    Returns a dict of per-row predictions for all 3 approaches.
    """
    windowed_texts = build_context_windows(texts, model.window_size)
    roberta_event_probs, roberta_highlight_scores = model.predict_probs(windowed_texts)

    ml_pred_event_id, ml_highlight_score = [], []
    lex_pred_event_id, lex_score = [], []
    hybrid_pred_event_id, hybrid_score_list = [], []

    for text, r_probs, r_h_score in zip(texts, roberta_event_probs, roberta_highlight_scores):
        # --- ML Model Only ---
        ml_event_id = max(range(len(r_probs)), key=lambda i: r_probs[i])
        ml_pred_event_id.append(ml_event_id)
        ml_highlight_score.append(r_h_score)

        # --- Lexicon Only ---
        l_prob = lexicon.score_chunk(text)
        lexicon_features = lexicon.generate_features(text)
        max_lex_score, lex_event_id = -1.0, 0
        for i in range(len(r_probs)):
            event_name = id2label.get(i, "normal_play")
            l_score = lexicon_features.get(event_name, 0.0)
            if l_score > max_lex_score:
                max_lex_score, lex_event_id = l_score, i
        lex_pred_event_id.append(lex_event_id)
        lex_score.append(l_prob)

        # --- Hybrid: highlight score (unchanged — still a blend, not gated) ---
        hybrid_score = min(1.0, (r_h_score * ROBERTA_WEIGHT) + (l_prob * HIGHLIGHT_LEXICON_WEIGHT))
        hybrid_score_list.append(hybrid_score)

        # --- Hybrid: event classification (confidence-gated) ---
        sorted_probs = sorted(r_probs, reverse=True)
        top1, top2 = sorted_probs[0], sorted_probs[1] if len(sorted_probs) > 1 else 0.0
        margin = top1 - top2

        if margin < MARGIN_THRESHOLD:
            # RoBERTa is uncertain here — let the lexicon weigh in.
            max_hybrid_prob, hyb_event_id = -1.0, 0
            for i, p in enumerate(r_probs):
                event_name = id2label.get(i, "normal_play")
                l_prob_for_event = lexicon_features.get(event_name, 0.0)
                l_prob_normalized = min(1.0, float(l_prob_for_event) * 100.0) * CATEGORY_WEIGHTS.get(event_name, 1.0)
                event_hybrid_prob = (p * ROBERTA_WEIGHT) + (l_prob_normalized * EVENT_LEXICON_WEIGHT)
                if event_hybrid_prob > max_hybrid_prob:
                    max_hybrid_prob, hyb_event_id = event_hybrid_prob, i
        else:
            # RoBERTa is already confident — keep its prediction untouched.
            hyb_event_id = ml_event_id

        hybrid_pred_event_id.append(hyb_event_id)

    return {
        "ML Model Only": {"event_id": ml_pred_event_id, "score": ml_highlight_score},
        "Lexicon Only": {"event_id": lex_pred_event_id, "score": lex_score},
        "Hybrid Model": {"event_id": hybrid_pred_event_id, "score": hybrid_score_list},
    }

In [10]:

# ============================================================
# CELL 10: Ground-truth extraction from a dataset CSV
#
# Handles both current ("event_class"/"highlight_score") and older
# ("event"/"score") column names, same fallback as
# build_windowed_dataframe() in colab_backend.py and _extract_labels()
# in the local pipeline's evaluator.py: missing/blank event -> normal_play.
#
# SCALE CHECK: highlight_score / score is expected to be a 0-1
# probability (predictions from the model are always in [0, 1] --
# sigmoid output, and Hybrid clips to min(1.0, ...)). If a CSV's score
# column isn't on that scale (leftover raw counts, a 0-100 percentage,
# an unrelated numeric column picked up by the "score" fallback, etc.)
# every row in that file will contribute a huge squared error once
# pooled with the rest -- pooled MSE/MAE is a weighted average of the
# per-match values, so it can never legitimately exceed the worst
# individual match. If COMBINED regression is much higher than every
# per-match regression you print, that mismatch is the cause, and this
# warning tells you which file and how far out of range it is.
# ============================================================
def load_dataset_csv(csv_path: str, label2id: dict):
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=["text"]).reset_index(drop=True)

    event_col = "event_class" if "event_class" in df.columns else "event"
    score_col = "highlight_score" if "highlight_score" in df.columns else "score"

    df["true_event_id"] = df[event_col].apply(
        lambda x: label2id.get(str(x).strip(), 0) if pd.notna(x) and str(x).strip() else 0
    )
    df["true_score"] = pd.to_numeric(df[score_col], errors="coerce").fillna(0.0)

    out_of_range = df[(df["true_score"] < 0.0) | (df["true_score"] > 1.0)]
    if len(out_of_range) > 0:
        match_name = os.path.splitext(os.path.basename(csv_path))[0]
        print(
            f"  [!] SCALE WARNING in '{match_name}': {len(out_of_range)}/{len(df)} rows have "
            f"{score_col}=out of the expected [0, 1] range "
            f"(min={out_of_range['true_score'].min():.4f}, max={out_of_range['true_score'].max():.4f}). "
            f"These rows will inflate combined MSE/MAE once pooled with other matches. "
            f"Check that '{score_col}' in this CSV is really a 0-1 highlight probability."
        )

    return df



In [11]:

# ============================================================
# CELL 11: Metrics computation
#
# Mirrors make_compute_metrics() in colab_backend.py / colab
# testing/colab_training.py -- sklearn weighted + macro averages over
# all classes (including normal_play), plus MSE/MAE for regression.
#
# VALID_SCORE_RANGE: highlight_score / score is a 0-1 probability by
# design (model outputs are always in [0,1]). Rows whose ground-truth
# true_score falls outside that range are excluded from regression
# metrics only (classification is untouched) -- they're bad ground
# truth, not model error, and including them makes MSE/MAE meaningless
# once pooled with clean matches. See the SCALE WARNING printed by
# load_dataset_csv() in Cell 10 for which file/rows are affected.
# ============================================================
VALID_SCORE_RANGE = (0.0, 1.0)


def filter_valid_scores(y_true, y_pred):
    """Drops (true, pred) pairs where y_true is outside VALID_SCORE_RANGE.
    Returns (filtered_true, filtered_pred, num_excluded)."""
    lo, hi = VALID_SCORE_RANGE
    filtered = [(t, p) for t, p in zip(y_true, y_pred) if lo <= t <= hi]
    num_excluded = len(y_true) - len(filtered)
    if filtered:
        f_true, f_pred = zip(*filtered)
        return list(f_true), list(f_pred), num_excluded
    return [], [], num_excluded


def compute_classification_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
    }


def compute_regression_metrics(y_true, y_pred):
    filtered_true, filtered_pred, num_excluded = filter_valid_scores(y_true, y_pred)
    if num_excluded > 0:
        print(
            f"  [i] Excluded {num_excluded}/{len(y_true)} rows from regression metrics "
            f"(true_score outside {VALID_SCORE_RANGE} -- bad ground truth, see Cell 10 warning)."
        )
    if not filtered_true:
        return {"mse": float("nan"), "mae": float("nan")}
    return {
        "mse": mean_squared_error(filtered_true, filtered_pred),
        "mae": mean_absolute_error(filtered_true, filtered_pred),
    }


def print_metrics_table(title, metrics_by_approach, keys):
    print(f"\n--- {title} ---")
    header = f"{'Approach':<16}" + "".join(f"{k:>16}" for k in keys)
    print(header)
    print("-" * len(header))
    for approach, metrics in metrics_by_approach.items():
        row = f"{approach:<16}" + "".join(f"{metrics[k]:>16.4f}" for k in keys)
        print(row)


def compute_per_class_metrics(y_true, y_pred, id2label):
    """Per-event breakdown (precision/recall/f1/support), the same shape as
    sklearn's classification_report -- a single pooled precision/recall/f1
    hides how the model does on any one event class, so this is reported
    alongside (not instead of) the macro/weighted summary."""
    labels = sorted(id2label.keys())
    target_names = [id2label[i] for i in labels]
    report = classification_report(
        y_true, y_pred, labels=labels, target_names=target_names,
        output_dict=True, zero_division=0,
    )
    return {name: report[name] for name in target_names}


def print_per_class_table(title, per_class_by_approach, event_names):
    print(f"\n--- {title}: Per-Event ---")
    header = f"{'Event':<14}{'precision':>12}{'recall':>12}{'f1-score':>12}{'support':>10}"
    for approach, per_class in per_class_by_approach.items():
        print(f"\n{approach}:")
        print(header)
        print("-" * len(header))
        for event in event_names:
            m = per_class[event]
            print(
                f"{event:<14}{m['precision']:>12.4f}{m['recall']:>12.4f}"
                f"{m['f1-score']:>12.4f}{m['support']:>10.0f}"
            )



In [12]:

# ============================================================
# CELL 12: Run evaluation over all uploaded datasets
#
# Computes per-match metrics for the reported approaches, then pools
# predictions across every match for a combined/aggregate result.
#
# APPROACH_NAMES controls what gets scored/printed/exported below.
# "Lexicon Only" is intentionally left out here -- only ML Model Only
# and Hybrid Model are reported. (run_all_approaches() still computes
# Lexicon Only internally in Cell 9 since Hybrid depends on it; it's
# just not included in this list.)
#
# OUTLIER DIAGNOSTIC: pooled_match_names tracks which match each pooled
# row came from, so if COMBINED regression comes out much worse than
# every per-match regression (a mathematical impossibility for a clean
# weighted average), we can print the worst-offending rows and see
# exactly which match/row is responsible, instead of guessing.
# ============================================================
CLASSIFICATION_KEYS = ["accuracy", "precision", "recall", "f1"]  # TODO: re-add precision_macro, recall_macro, f1_macro
REGRESSION_KEYS = ["mse", "mae"]
APPROACH_NAMES = ["ML Model Only", "Hybrid Model"]

EVENT_NAMES = [loaded_model.id2label[i] for i in sorted(loaded_model.id2label.keys())]

per_match_results = {}  # match_name -> {"classification": {...}, "per_class": {...}, "regression": {...}}
pooled_true_event, pooled_true_score = [], []
pooled_pred_event = {name: [] for name in APPROACH_NAMES}
pooled_pred_score = {name: [] for name in APPROACH_NAMES}
pooled_match_names = []  # parallel to pooled_true_event / pooled_true_score

for csv_path in DATASET_CSV_PATHS:
    match_name = os.path.splitext(os.path.basename(csv_path))[0]
    print(f"\n{'=' * 60}\nEvaluating: {match_name}\n{'=' * 60}")

    df = load_dataset_csv(csv_path, loaded_model.label2id)
    texts = df["text"].tolist()
    true_event = df["true_event_id"].tolist()
    true_score = df["true_score"].tolist()

    predictions = run_all_approaches(texts, loaded_model.id2label, loaded_model, lexicon_model)

    classification_metrics, per_class_metrics, regression_metrics = {}, {}, {}
    for approach in APPROACH_NAMES:
        pred_event = predictions[approach]["event_id"]
        pred_score = predictions[approach]["score"]

        classification_metrics[approach] = compute_classification_metrics(true_event, pred_event)
        per_class_metrics[approach] = compute_per_class_metrics(true_event, pred_event, loaded_model.id2label)
        regression_metrics[approach] = compute_regression_metrics(true_score, pred_score)

        pooled_pred_event[approach].extend(pred_event)
        pooled_pred_score[approach].extend(pred_score)

    pooled_true_event.extend(true_event)
    pooled_true_score.extend(true_score)
    pooled_match_names.extend([match_name] * len(true_score))

    print_metrics_table("Classification", classification_metrics, CLASSIFICATION_KEYS)
    print_per_class_table("Classification", per_class_metrics, EVENT_NAMES)
    print_metrics_table("Regression", regression_metrics, REGRESSION_KEYS)

    per_match_results[match_name] = {
        "classification": classification_metrics,
        "per_class": per_class_metrics,
        "regression": regression_metrics,
    }

# --- Combined (pooled across all uploaded matches) ---
print(f"\n{'=' * 60}\nCOMBINED (all {len(DATASET_CSV_PATHS)} matches pooled)\n{'=' * 60}")
combined_classification, combined_per_class, combined_regression = {}, {}, {}
for approach in APPROACH_NAMES:
    combined_classification[approach] = compute_classification_metrics(
        pooled_true_event, pooled_pred_event[approach]
    )
    combined_per_class[approach] = compute_per_class_metrics(
        pooled_true_event, pooled_pred_event[approach], loaded_model.id2label
    )
    combined_regression[approach] = compute_regression_metrics(
        pooled_true_score, pooled_pred_score[approach]
    )

print_metrics_table("Classification", combined_classification, CLASSIFICATION_KEYS)
print_per_class_table("Classification", combined_per_class, EVENT_NAMES)
print_metrics_table("Regression", combined_regression, REGRESSION_KEYS)

# --- Sanity check: pooled MSE/MAE can never legitimately exceed the
# worst per-match MSE/MAE (it's a weighted average of them). If it
# does, print the worst-offending rows so you can trace the bad match.
for approach in APPROACH_NAMES:
    worst_per_match_mse = max(
        per_match_results[m]["regression"][approach]["mse"] for m in per_match_results
    )
    combined_mse = combined_regression[approach]["mse"]
    if combined_mse > worst_per_match_mse * 1.5:
        print(
            f"\n[!] '{approach}': COMBINED MSE ({combined_mse:.4f}) is well above the worst "
            f"per-match MSE ({worst_per_match_mse:.4f}) -- this points to a score-scale mismatch "
            f"in the pooled data, not a real regression. Top offending rows:"
        )
        pred_score_list = pooled_pred_score[approach]
        errors = [
            (abs(t - p), i, m, t, p)
            for i, (t, p, m) in enumerate(zip(pooled_true_score, pred_score_list, pooled_match_names))
        ]
        errors.sort(reverse=True)
        print(f"{'match':<30}{'row':>6}{'true_score':>14}{'pred_score':>14}{'abs_error':>14}")
        for abs_err, idx, m, t, p in errors[:10]:
            print(f"{m:<30}{idx:>6}{t:>14.4f}{p:>14.4f}{abs_err:>14.4f}")




Evaluating: dataset_match_1_Geo_vs_POR_WCA_2023 (1)

--- Classification ---
Approach                accuracy       precision          recall              f1
--------------------------------------------------------------------------------
ML Model Only             0.8775          0.8889          0.8775          0.8675
Hybrid Model              0.8790          0.8902          0.8790          0.8694

--- Classification: Per-Event ---

ML Model Only:
Event            precision      recall    f1-score   support
------------------------------------------------------------
normal_play         0.8915      0.9628      0.9258      2151
try                 0.9630      0.6500      0.7761        40
goal_kick           0.9623      0.4215      0.5862       121
card_event          0.0000      0.0000      0.0000         0
scrum               0.9515      0.6577      0.7778       149
maul                0.9394      0.7045      0.8052        44
lineout             0.5740      0.8739      0.6929       111

In [ ]:

# ============================================================
# CELL 13: Export results as JSON
#
# Schema mirrors commentary_analysis_system/data/output/evaluation_metrics.json
# ("Lexicon Only" / "ML Model Only" / "Hybrid Model" keys), extended
# with accuracy/precision_macro/recall_macro/f1_macro plus a per-event
# ("per_class") breakdown for each of the 9 event classes. Includes
# both the per-match breakdown and the combined/pooled result.
# ============================================================
from datetime import datetime, timezone

output = {
    "model_dir": os.path.basename(MODEL_ZIP_PATH),
    "datasets": DATASET_CSV_PATHS,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "hybrid_weights": {
        "highlight_lexicon_weight": HIGHLIGHT_LEXICON_WEIGHT,
        "event_lexicon_weight": EVENT_LEXICON_WEIGHT,
        "roberta_weight": ROBERTA_WEIGHT,
    },
    "per_match": per_match_results,
    "combined": {
        "classification": combined_classification,
        "per_class": combined_per_class,
        "regression": combined_regression,
    },
}

OUTPUT_PATH = "/content/evaluation_results.json"
with open(OUTPUT_PATH, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved results to {OUTPUT_PATH}")
files.download(OUTPUT_PATH)


Saved results to /content/evaluation_results.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>